In [2]:
import os
import pandas as pd

# File setup
file_path = os.path.expanduser("~/Documents/Research/diagnosis_data.csv")
chunk_size = 100000
n_rows = 500000

data_list = []
rows_read = 0

# Read CSV in chunks
for chunk in pd.read_csv(file_path, chunksize=chunk_size):
    data_list.append(chunk)
    rows_read += len(chunk)

    # Print column names from the first chunk only
    if rows_read == len(chunk):
        print("Column headers:", chunk.columns.tolist())
        
    if rows_read >= n_rows:
        break

# Combine all chunks into one DataFrame
combined_data = pd.concat(data_list, ignore_index=True)

# Calculate percentages
total_count = combined_data["Female"].notna().sum()  # count of valid values
female_count = combined_data["Female"].sum()         # sum of 1s = number of females
male_count = total_count - female_count              # rest are males

female_percentage = (female_count / total_count) * 100
male_percentage = (male_count / total_count) * 100

print(f"Female: {female_percentage:.2f}%")
print(f"Male: {male_percentage:.2f}%")


Column headers: ['Age', 'Female', 'K088', 'Pay1', 'NCHS', 'Total Cost', 'ZIP', 'Diagnosis']
Female: 55.66%
Male: 44.34%


In [10]:
import pandas as pd
import os

# File path and chunk settings
file_path = os.path.expanduser("~/Documents/Research/diagnosis_data.csv")
chunk_size = 100000
n_rows = 500000

# Initialize an empty list to store cleaned data
data_list = []
rows_read = 0

# Read file in chunks
for chunk in pd.read_csv(file_path, chunksize=chunk_size):
    # Remove rows with negative Pay1 values
    chunk = chunk[chunk['Pay1'] >= 0]

    # Append to the list
    data_list.append(chunk)

    rows_read += len(chunk)
    if rows_read >= n_rows:
        break

# Combine all chunks into a single DataFrame
df = pd.concat(data_list, ignore_index=True)

### Step 1: % of Females in each Diagnosis type
diagnosis_counts = df.groupby('Diagnosis')['Female'].agg(['count', 'sum'])
diagnosis_counts['Female_Percentage'] = 100 * diagnosis_counts['sum'] / diagnosis_counts['count']

# Optional: round the results
diagnosis_female_percent = diagnosis_counts[['Female_Percentage']].round(2)
print("Step 1: Percentage of Females by Diagnosis:")
print(diagnosis_female_percent)

### Step 2: % of Pay1 (1 to 6) values in each Diagnosis
# Filter Pay1 to only include values from 1 to 6
df = df[df['Pay1'].between(1, 6)]

# Get count of each Pay1 per Diagnosis
pay1_counts = df.groupby(['Diagnosis', 'Pay1']).size().unstack(fill_value=0)

# Convert counts to percentages row-wise
pay1_percent = pay1_counts.div(pay1_counts.sum(axis=1), axis=0) * 100
pay1_percent = pay1_percent.round(2)

print("\nStep 2: Percentage of Pay1 values (1-6) by Diagnosis:")
print(pay1_percent)


Step 1: Percentage of Females by Diagnosis:
           Female_Percentage
Diagnosis                   
BLD                    57.39
CIR                    52.09
DEN                    51.42
DIG                    55.56
EAR                    52.40
END                    53.34
EXT                    45.41
EYE                    52.46
FAC                    50.22
GEN                    68.40
INF                    50.66
INJ                    49.33
MAL                    43.21
MBD                    44.39
MUS                    56.49
NEO                    54.56
NVS                    59.06
PNL                    49.24
PRG                   100.00
RSP                    54.57
SKN                    49.79
SYM                    59.45

Step 2: Percentage of Pay1 values (1-6) by Diagnosis:
Pay1           1      2      3      4     5      6
Diagnosis                                         
BLD        41.44  18.82  27.38   4.20  1.02   7.13
CIR        46.28  14.17  30.25   4.49  0.89   3.93
D

In [11]:
import pandas as pd
import os

# File path and chunk settings
file_path = os.path.expanduser("~/Documents/Research/diagnosis_data.csv")
chunk_size = 100000
n_rows = 500000

# Initialize an empty list to store cleaned data
data_list = []
rows_read = 0

# Read file in chunks
for chunk in pd.read_csv(file_path, chunksize=chunk_size):
    # Remove rows with any negative values in Pay1, ZIP, or NCHS
    chunk = chunk[(chunk['Pay1'] >= 0) & (chunk['ZIP'] >= 0) & (chunk['NCHS'] >= 0)]

    data_list.append(chunk)
    rows_read += len(chunk)
    if rows_read >= n_rows:
        break

# Combine chunks
df = pd.concat(data_list, ignore_index=True)

### Step 1: % of Females in each Diagnosis
diagnosis_counts = df.groupby('Diagnosis')['Female'].agg(['count', 'sum'])
diagnosis_counts['Female_Percentage'] = 100 * diagnosis_counts['sum'] / diagnosis_counts['count']
diagnosis_female_percent = diagnosis_counts[['Female_Percentage']].round(2)
print("Step 1: % of Females by Diagnosis:")
print(diagnosis_female_percent)

### Step 2: % of Pay1 (1 to 6) in each Diagnosis
df_pay1 = df[df['Pay1'].between(1, 6)]
pay1_counts = df_pay1.groupby(['Diagnosis', 'Pay1']).size().unstack(fill_value=0)
pay1_percent = pay1_counts.div(pay1_counts.sum(axis=1), axis=0) * 100
pay1_percent = pay1_percent.round(2)
print("\nStep 2: % of Pay1 values (1–6) by Diagnosis:")
print(pay1_percent)

### Step 3: % of ZIP (1 to 4) in each Diagnosis
df_zip = df[df['ZIP'].between(1, 4)]
zip_counts = df_zip.groupby(['Diagnosis', 'ZIP']).size().unstack(fill_value=0)
zip_percent = zip_counts.div(zip_counts.sum(axis=1), axis=0) * 100
zip_percent = zip_percent.round(2)
print("\nStep 3: % of ZIP values (1–4) by Diagnosis:")
print(zip_percent)

### Step 4: % of NCHS (1 to 6) in each Diagnosis
df_nchs = df[df['NCHS'].between(1, 6)]
nchs_counts = df_nchs.groupby(['Diagnosis', 'NCHS']).size().unstack(fill_value=0)
nchs_percent = nchs_counts.div(nchs_counts.sum(axis=1), axis=0) * 100
nchs_percent = nchs_percent.round(2)
print("\nStep 4: % of NCHS values (1–6) by Diagnosis:")
print(nchs_percent)


Step 1: % of Females by Diagnosis:
           Female_Percentage
Diagnosis                   
BLD                    57.34
CIR                    52.13
DEN                    51.43
DIG                    55.61
EAR                    52.39
END                    53.38
EXT                    45.35
EYE                    52.59
FAC                    50.24
GEN                    68.41
INF                    50.69
INJ                    49.34
MAL                    43.12
MBD                    44.39
MUS                    56.49
NEO                    54.54
NVS                    59.03
PNL                    49.29
PRG                   100.00
RSP                    54.59
SKN                    49.79
SYM                    59.43

Step 2: % of Pay1 values (1–6) by Diagnosis:
Pay1           1      2      3      4     5      6
Diagnosis                                         
BLD        41.43  18.78  27.41   4.20  1.02   7.16
CIR        46.38  14.13  30.22   4.45  0.89   3.93
DEN        11.07  3